# Circular Splines

A kingdon translation of the ganja.js [circular splines example](https://enki.ws/ganja.js/examples/coffeeshop.html#PAHOy-VKwv). Drag the points around, and do not forget to switch the dimension from 2DPGA to 3DPGA!

In [1]:
import numpy as np
from kingdon import Algebra, UPoint, Point, EVector, Direction

In [2]:
alg = Algebra(2, 0, 1, full_layout=True, extra_types=[UPoint, Point, EVector, Direction], values_asarray=np.asarray)

Logarithm and exponential of a 2D PGA rotor.

In [3]:
def log(R):
    return (B := R.grade(2)) / np.sinc(np.arctan2(B.normsq().e ** 0.5, R.e) / np.pi)

# kingdon's own .exp() does not support numpy arrays yet.
def exp(B):
    return np.cos(l := B.normsq().e ** 0.5) + B * np.sinc(l / np.pi)

Number of points, rotor orbit.

In [4]:
n = 5
t = np.linspace(0, 1, 64)
edges = np.stack([np.arange(63), np.arange(1, 64)], axis=-1)  # Index pairs of consecutive points.

def orbit(R, p):
    # Sandwich a copy of p for every t, and connect consecutive points into a line cloud of shape (63, 2).
    return (exp(t * log(R)) >> p[None])[edges]

Our set of points, and our seed point.

In [5]:
p = [alg.upoint(e1=-1.5 + i * 3 / (n - 1), e2=0.6 * np.random.random(), e3=0).dual() 
     for i in range(n)]
c0 = p[0] - alg.evector(e1=0.25, e2=0.25, e3=0).dual()

Calculate the circular spline.

In [6]:
def circular_spline(p, c0):
    R = []
    for i in range(len(p) - 1):
        # Line through previous center and current spline point.
        L1 = log(R[i - 1] if i else p[0] * ~c0) & ~p[i]
        # Line between two consecutive spline points.
        L2 = (p[i + 1] & p[i]) | (p[i] + p[i + 1])
        # Rotor we need.
        R.append((L2 * L1).normalized())
    return R

In [7]:
def graph_func():
    R = circular_spline(p, c0)
    return [
        0xff9900, *[x for i, pi in enumerate(p) for x in (pi, f'p{i}')],
        0x0e94fb, *[orbit(Ri, pi) for Ri, pi in zip(R, p)], c0,
    ]

alg.graph(graph_func, lineWidth=3, pointRadius=2, grid=True, scale=1.5)